In [1]:
import jax
import jax.numpy as jnp
from jax import jit, vmap, jacfwd
import jax.nn.initializers as inits
import optax
import flax.linen as nn
import numpy as np
import pandas as pd

# Enable 64-bit precision for stable gradients
jax.config.update("jax_enable_x64", True)
print("JAX is using:", jax.devices())

JAX is using: [CpuDevice(id=0)]


In [2]:
# Physics constants
MU = 1.0
KAPPA = 10.0
PULL_STEPS = jnp.array([0.001, 0.002, 0.005, 0.01, 0.05, 0.10])
BC_WEIGHT = 100.0  # BC penalty weight

In [ ]:
# Training parameters
LEARNING_RATE = 1e-3
EPOCHS_INITIAL = 3000  # Longer training for the first pull
EPOCHS_STEP = 1000     # Shorter training for subsequent pulls
KEY = jax.random.PRNGKey(42)
OUTPUT_FILE = '/Users/murat/Desktop/DEM-FEA-complex-geometry-comparison/deep_energy_method_results.csv'


In [ ]:
# DEM Model 
class DeepEnergyModel(nn.Module):
    @nn.compact
    def __call__(self, x):
        # Normalize inputs based on global domain bounds (max x=1.0, max y=1.0)
        x_norm = x / jnp.array([1.0, 1.0])
        h = (x_norm - 0.5) * 2.0 
        
        h = nn.tanh(nn.Dense(128)(h))
        h = nn.tanh(nn.Dense(128)(h))
        h = nn.tanh(nn.Dense(128)(h))
        
        return nn.Dense(2, kernel_init=inits.zeros, bias_init=inits.zeros)(h)

# Initialize global model
dem_model = DeepEnergyModel()

In [5]:
# Geometries
def generate_domain_and_boundaries(shape_type, spacing=0.02):
    """Generates dense collocation points for integration and boundary points."""
    
    # Define bounding box based on shape
    if shape_type in ['rectangle', 'rectangle_hole']:
        xx, yy = np.meshgrid(np.arange(0, 1.0 + spacing, spacing), 
                             np.arange(0, 0.5 + spacing, spacing))
    elif shape_type in ['l_shape', 'l_shape_hole']:
        xx, yy = np.meshgrid(np.arange(0, 1.0 + spacing, spacing), 
                             np.arange(0, 1.0 + spacing, spacing))
    else:
        raise ValueError(f"Unknown shape type: {shape_type}")
        
    pts = np.vstack([xx.ravel(), yy.ravel()]).T
    
    # Apply Masks
    if shape_type == 'rectangle':
        mask = np.ones(len(pts), dtype=bool)
    elif shape_type == 'rectangle_hole':
        mask = (pts[:, 0] - 0.5)**2 + (pts[:, 1] - 0.25)**2 > 0.1**2
    elif shape_type == 'l_shape':
        mask = ~((pts[:, 0] > 0.5 + 1e-5) & (pts[:, 1] > 0.5 + 1e-5))
    elif shape_type == 'l_shape_hole':
        l_mask = ~((pts[:, 0] > 0.5 + 1e-5) & (pts[:, 1] > 0.5 + 1e-5))
        hole_mask = (pts[:, 0] - 0.25)**2 + (pts[:, 1] - 0.25)**2 > 0.1**2
        mask = l_mask & hole_mask
        
    domain_pts = pts[mask]
    
    # Uniform weight for numerical integration (Area per point)
    domain_wts = np.ones(len(domain_pts)) * (spacing * spacing)
    
    # Identify Boundary Points
    tol = 1e-5
    left_bc_pts = domain_pts[domain_pts[:, 0] <= tol]
    right_bc_pts = domain_pts[domain_pts[:, 0] >= 1.0 - tol]
    
    return jnp.array(domain_pts), jnp.array(domain_wts), jnp.array(left_bc_pts), jnp.array(right_bc_pts)

In [ ]:
def network_displacement(params, x):
    return dem_model.apply(params, x)

In [7]:
def point_energy(params, x):
    # F = I + Grad(u)
    grad_u = jacfwd(network_displacement, argnums=1)(params, x)
    F = jnp.eye(2) + grad_u
    
    # Right Cauchy-Green
    C = F.T @ F 
    I1 = jnp.trace(C) 
    J = jnp.linalg.det(F)
    
    # Prevent NaN during wild network initializations
    J_safe = jnp.clip(J, a_min=1e-5, a_max=None)
    
    # Neo-Hookean Compressible
    W = (MU / 2.0) * (I1 - 2.0) - MU * jnp.log(J_safe) + (KAPPA / 2.0) * (jnp.log(J_safe))**2
    
    # Penalty for inversion
    penalty = jnp.where(J <= 1e-5, 1e6 * (1e-5 - J)**2, 0.0)
    
    return W + penalty

In [8]:
def point_stress(params, x):
    # 1. Kinematics
    grad_u = jacfwd(network_displacement, argnums=1)(params, x)
    F = jnp.eye(2) + grad_u
    J = jnp.linalg.det(F)
    J_safe = jnp.clip(J, a_min=1e-5, a_max=None)
    
    # 2. Left Cauchy-Green tensor
    B = F @ F.T
    I = jnp.eye(2)
    
    # 3. Cauchy Stress (Analytical derivation for the given Neo-Hookean W)
    sigma = (1.0 / J_safe) * (MU * (B - I) + KAPPA * jnp.log(J_safe) * I)
    
    sig_xx = sigma[0, 0]
    sig_yy = sigma[1, 1]
    sig_xy = sigma[0, 1] 
    
    # 4. von Mises Stress (Using 2D Plane Stress formulation)
    von_mises = jnp.sqrt(sig_xx**2 - sig_xx*sig_yy + sig_yy**2 + 3.0*sig_xy**2)
    
    return sig_xx, sig_yy, sig_xy, von_mises

In [9]:
batch_displacement = vmap(network_displacement, in_axes=(None, 0))
batch_energy = vmap(point_energy, in_axes=(None, 0))
batch_stress = vmap(point_stress, in_axes=(None, 0)) # Vectorized stress

In [10]:
def build_loss_fn(domain_pts, domain_wts, left_bc_pts, right_bc_pts):
    """Creates a compiled loss function for a specific shape."""
    
    def loss_fn(params, current_pull):
        # 1. Internal Energy (Quadrature Integral)
        W_all = batch_energy(params, domain_pts)
        W_integral = jnp.sum(W_all * domain_wts)
        
        # 2. Left Boundary Penalty (Fixed: u_x = 0, u_y = 0)
        u_left = batch_displacement(params, left_bc_pts)
        left_error = jnp.mean(jnp.sum(u_left**2, axis=1))
        
        # 3. Right Boundary Penalty (Pulled: u_x = pull, u_y = 0)
        u_right = batch_displacement(params, right_bc_pts)
        right_targets = jnp.hstack([jnp.ones((len(right_bc_pts), 1)) * current_pull, 
                                    jnp.zeros((len(right_bc_pts), 1))])
        right_error = jnp.mean(jnp.sum((u_right - right_targets)**2, axis=1))
        
        # Total Loss
        return W_integral + BC_WEIGHT * (left_error + right_error)
        
    # Idiomatic JAX: jit the value_and_grad combination
    return jit(jax.value_and_grad(loss_fn))

In [11]:
def solve_shape(shape_name):
    print(f"\n--- Initializing {shape_name} ---")
    domain_pts, domain_wts, left_bc_pts, right_bc_pts = generate_domain_and_boundaries(shape_name)
    print(f"Nodes generated: {len(domain_pts)}")
    
    # Initialize network weights
    params = dem_model.init(KEY, jnp.ones((2,)))
    optimizer = optax.adam(LEARNING_RATE)
    opt_state = optimizer.init(params)
    
    # Build compiled loss function for this shape
    loss_and_grad_fn = build_loss_fn(domain_pts, domain_wts, left_bc_pts, right_bc_pts)
    
    shape_results = []
    
    # Load Stepping / Continuation
    for step_idx, current_pull in enumerate(PULL_STEPS):
        print(f"Training pull step: {current_pull}...")
        current_epochs = EPOCHS_INITIAL if step_idx == 0 else EPOCHS_STEP
        
        for epoch in range(current_epochs):
            loss_val, grads = loss_and_grad_fn(params, current_pull)
            updates, opt_state = optimizer.update(grads, opt_state)
            params = optax.apply_updates(params, updates)
            
            if epoch % 500 == 0 or epoch == current_epochs - 1:
                print(f"  Epoch {epoch:04d}/{current_epochs} | Loss: {loss_val:.6f}")
                
        # Extract full field displacements and stresses
        u_domain = batch_displacement(params, domain_pts)
        sig_xx, sig_yy, sig_xy, von_mises = batch_stress(params, domain_pts)
        
        # Format for dataframe
        for i in range(len(domain_pts)):
            shape_results.append({
                'Shape': shape_name,
                'Pull': float(current_pull),
                'X': float(domain_pts[i, 0]),
                'Y': float(domain_pts[i, 1]),
                'u_x': float(u_domain[i, 0]),
                'u_y': float(u_domain[i, 1]),
                'sigma_xx': float(sig_xx[i]),
                'sigma_yy': float(sig_yy[i]),
                'sigma_xy': float(sig_xy[i]),
                'von_mises': float(von_mises[i])
            })
            
    return shape_results

In [13]:
import os
import pandas as pd

shapes = ['rectangle', 'rectangle_hole', 'l_shape', 'l_shape_hole']
all_results = []
    
for shape in shapes:
    results = solve_shape(shape)
    all_results.extend(results)
        
df = pd.DataFrame(all_results)

# --- NEW CODE: Force create the directory structure ---
output_dir = os.path.dirname(OUTPUT_FILE)
if output_dir:  # Ensure it's not an empty string
    os.makedirs(output_dir, exist_ok=True)
# ----------------------------------------------------

df.to_csv(OUTPUT_FILE, index=False)
print(f"\nTraining complete! Extracted {len(df)} nodal data points to '{OUTPUT_FILE}'.")


--- Initializing rectangle ---
Nodes generated: 1326
Training pull step: 0.001...
  Epoch 0000/3000 | Loss: 0.000100
  Epoch 0500/3000 | Loss: 0.000001
  Epoch 1000/3000 | Loss: 0.000001
  Epoch 1500/3000 | Loss: 0.000001
  Epoch 2000/3000 | Loss: 0.000001
  Epoch 2500/3000 | Loss: 0.000001
  Epoch 2999/3000 | Loss: 0.000001
Training pull step: 0.002...
  Epoch 0000/1000 | Loss: 0.000103
  Epoch 0500/1000 | Loss: 0.000005
  Epoch 0999/1000 | Loss: 0.000004
Training pull step: 0.005...
  Epoch 0000/1000 | Loss: 0.000918
  Epoch 0500/1000 | Loss: 0.000028
  Epoch 0999/1000 | Loss: 0.000027
Training pull step: 0.01...
  Epoch 0000/1000 | Loss: 0.002582
  Epoch 0500/1000 | Loss: 0.000124
  Epoch 0999/1000 | Loss: 0.000117
Training pull step: 0.05...
  Epoch 0000/1000 | Loss: 0.161074
  Epoch 0500/1000 | Loss: 0.002797
  Epoch 0999/1000 | Loss: 0.002725
Training pull step: 0.1...
  Epoch 0000/1000 | Loss: 0.258124
  Epoch 0500/1000 | Loss: 0.010468
  Epoch 0999/1000 | Loss: 0.010324

--- I